In [ ]:
!pip install datasets

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

In [ ]:
from pathlib import Path

# Data and checkpoints live in the repo (no Google Drive mount needed).
DATA_DIR = Path("..") / "data" / "raw"
OUTPUT_DIR = Path("..") / "outputs" / "dcard_custom"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR.resolve())
print("OUTPUT_DIR =", OUTPUT_DIR.resolve())


In [ ]:
# Sanity check: expected CSV files must exist before training.
train_csv = DATA_DIR / "dcard_self_collected.csv"
assert train_csv.exists(), f"Missing training CSV: {train_csv}"
print("Found:", train_csv.name)


In [ ]:
dataset = load_dataset("csv", data_files=str(train_csv), split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset


In [ ]:

datasets = dataset.train_test_split(test_size=0.1)
datasets

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 5923
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 659
    })
})

In [ ]:

import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

def process_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples

tokenized_datasets = datasets.map(process_function, batched=True, remove_columns=datasets["train"].column_names)
tokenized_datasets

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/5923 [00:00<?, ? examples/s]

Map:   0%|          | 0/659 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5923
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 659
    })
})

In [ ]:


model = AutoModelForSequenceClassification.from_pretrained("bert-base-chinese")



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
!pip install evaluate

In [ ]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
auc_metric = evaluate.load("roc_auc")

In [ ]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict

    # 如果预测结果是多个类别的概率，取正类的概率
    if predictions.ndim > 1 and predictions.shape[1] > 1:
        probabilities = predictions[:, 1]
    else:
        probabilities = predictions

    # 根据概率计算预测类别
    predicted_classes = probabilities.argmax(axis=-1) if probabilities.ndim > 1 else (probabilities >= 0.5).astype(int)

    # 计算各项指标
    acc = acc_metric.compute(predictions=predicted_classes, references=labels)
    f1 = f1_metric.compute(predictions=predicted_classes, references=labels)
    auc = auc_metric.compute(prediction_scores=probabilities, references=labels)

    # 更新字典，将F1得分和AUC添加进去
    acc.update(f1)
    acc["auc"] = auc["roc_auc"]

    return acc

In [ ]:
!pip install transformers[torch]

In [ ]:
import sys
print(sys.path)

['/content', '/env/python', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/usr/local/lib/python3.10/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.10/dist-packages/IPython/extensions', '/usr/local/lib/python3.10/dist-packages/setuptools/_vendor', '/root/.ipython', '/root/.cache/huggingface/modules']


In [ ]:
!pip install accelerate -U

In [ ]:
from transformers import TrainingArguments

In [ ]:
pip install transformers[torch]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
pip install accelerate

In [ ]:
!pip install pytorch-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 4.1 MB/s eta 0:00:00


In [ ]:
!pip install transformers[torch]
!pip install accelerate -U

In [ ]:
train_args = TrainingArguments(output_dir=str(OUTPUT_DIR), # 输出文件夹
                               per_device_train_batch_size=128,  # 训练时的batch_size
                               per_device_eval_batch_size=128,  # 验证时的batch_size
                               logging_steps=10,                # log 打印的频率
                               evaluation_strategy="epoch",     # 评估策略
                               save_strategy="epoch",           # 保存策略
                               save_total_limit=5,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               num_train_epochs=5.0,
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args


In [ ]:
!pip install --upgrade transformers
!pip install --upgrade datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 54.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.42.4
    Uninstalling transformers-4.42.4:
      Successfully uninstalled transformers-4.42.4


In [ ]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model,
                  args=train_args,
                  train_dataset=tokenized_datasets["train"],
                  eval_dataset=tokenized_datasets["test"],
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

In [ ]:
trainer.evaluate(tokenized_datasets["test"])###訓練前

{'eval_loss': 0.7634400129318237,
 'eval_accuracy': 0.47040971168437024,
 'eval_f1': 0.5598991172761665,
 'eval_auc': 0.4353142577647347,
 'eval_runtime': 5.6999,
 'eval_samples_per_second': 115.615,
 'eval_steps_per_second': 1.053}

In [ ]:
trainer.train()###訓練中

Epoch,Training Loss,Validation Loss,Accuracy,F1,Auc
1,0.556800,0.495890,0.702580,0.687898,0.831819
2,0.421900,0.442983,0.778452,0.786550,0.874560
3,0.345600,0.438459,0.731411,0.718601,0.884512
4,0.289500,0.436635,0.816388,0.829337,0.890654
5,0.255400,0.444686,0.802731,0.813754,0.890842


TrainOutput(global_step=235, training_loss=0.3795727557324349, metrics={'train_runtime': 666.8174, 'train_samples_per_second': 44.412, 'train_steps_per_second': 0.352, 'total_flos': 1947702712297500.0, 'train_loss': 0.3795727557324349, 'epoch': 5.0})

In [ ]:
trainer.evaluate(tokenized_datasets["test"])

{'eval_loss': 0.43663522601127625,
 'eval_accuracy': 0.8163884673748103,
 'eval_f1': 0.8293370944992947,
 'eval_auc': 0.8906541001185169,
 'eval_runtime': 4.4028,
 'eval_samples_per_second': 149.676,
 'eval_steps_per_second': 1.363,
 'epoch': 5.0}

In [ ]:
trainer.predict(tokenized_datasets["test"]) ###訓練後灌入dcard驗證集

PredictionOutput(predictions=array([[-1.1995168 ,  0.97844726],
       [-2.1479    ,  1.6493165 ],
       [ 1.2044674 , -0.80683714],
       ...,
       [ 0.4851212 , -0.42032903],
       [-2.3534281 ,  1.7450349 ],
       [ 0.93531203, -0.4925237 ]], dtype=float32), label_ids=array([0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0,
       0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1,
       0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0,
       1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1